Script to compile images from a local folder, organize by metadata timestamp, and compile  
into monthly time lapse videos.

In [1]:
photo_folder = r"M:\Shared drives\UGS_Flux_Archive\Data_Downloads_Archive\Dugout_Ranch"
stationid = 'US-UTD' # used in naming the output 
file_identifier = "cam*.jpg" # string to search for within photo_folder to ID photos

In [2]:
from pathlib import Path
from datetime import datetime
from collections import defaultdict

In [ ]:
base_dir = Path(photo_folder)

monthly_photos = defaultdict(list)

# 1. Scan and group files by their metadata timestamp
for file in base_dir.rglob(file_identifier):
    # st_mtime is the safest cross-platform bet for when the file was saved
    timestamp = file.stat().st_mtime
    date_obj = datetime.fromtimestamp(timestamp)
    
    # Create a string key like "2026-05"
    month_key = date_obj.strftime("%Y-%m")
    
    # Store a tuple of (timestamp, file_path) so we can sort it accurately
    monthly_photos[month_key].append((timestamp, file))

# 2. Sort the files within each month chronologically
for month_key in monthly_photos:
    # Sort by the timestamp element (index 0 of the tuple)
    monthly_photos[month_key].sort(key=lambda x: x[0])
    
    # Overwrite with just the sorted file paths, removing the temporary timestamp
    monthly_photos[month_key] = [file for timestamp, file in monthly_photos[month_key]]
    
    print(f"Found {len(monthly_photos[month_key])} photos for {month_key}")

Found 386 photos for 2026-02
Found 465 photos for 2026-03
Found 462 photos for 2025-05
Found 728 photos for 2025-06
Found 474 photos for 2025-07
Found 166 photos for 2025-08
Found 494 photos for 2025-09
Found 1007 photos for 2025-10
Found 59 photos for 2025-11
Found 440 photos for 2025-12
Found 149 photos for 2026-01


In [18]:
import cv2

FPS = 10 

for month_key, files in monthly_photos.items():
    if not files:
        continue
        
    print(f"Processing timelapse for {month_key}...")
    
    # Read the first image to automatically detect the resolution (width/height)
    first_img = cv2.imread(str(files[0]))
    height, width, _ = first_img.shape
    video_resolution = (width, height)
    
    # Setup the video writer (Output filename, Codec, Frames Per Second, Resolution)
    output_filename = f"{stationid}_{month_key}.mp4"
    fourcc = cv2.VideoWriter_fourcc(*'mp4v') 
    video_writer = cv2.VideoWriter(output_filename, fourcc, FPS, video_resolution)
    
    # Write each image into the video file
    for file_path in files:
        img = cv2.imread(str(file_path))
        
        # Optional: If your photos vary in size, uncomment the line below to resize them
        # img = cv2.resize(img, video_resolution)
        
        video_writer.write(img)
        
    # Close the video file stream
    video_writer.release()
    print(f"Successfully saved {output_filename}!\n")

print("All timelapses completed.")

Processing timelapse for 2026-02...
Successfully saved US-UTD_2026-02.mp4!

Processing timelapse for 2026-03...
Successfully saved US-UTD_2026-03.mp4!

Processing timelapse for 2025-05...
Successfully saved US-UTD_2025-05.mp4!

Processing timelapse for 2025-06...
Successfully saved US-UTD_2025-06.mp4!

Processing timelapse for 2025-07...
Successfully saved US-UTD_2025-07.mp4!

Processing timelapse for 2025-08...
Successfully saved US-UTD_2025-08.mp4!

Processing timelapse for 2025-09...
Successfully saved US-UTD_2025-09.mp4!

Processing timelapse for 2025-10...
Successfully saved US-UTD_2025-10.mp4!

Processing timelapse for 2025-11...
Successfully saved US-UTD_2025-11.mp4!

Processing timelapse for 2025-12...
Successfully saved US-UTD_2025-12.mp4!

Processing timelapse for 2026-01...
Successfully saved US-UTD_2026-01.mp4!

All timelapses completed.
